# Poster Appendix: EMA and Guidance Ablations

This notebook collects the small ablation results that are useful as poster backup material:

1. **EMA length / post-hoc EMA target**: does changing EMA smoothing improve one-point PDFs or power spectra?
2. **CFG dropout + guidance scale**: does classifier-free guidance improve the conditional cosmology calibration?

The notebook is intentionally lightweight. It reads completed CSV/PNG outputs from the existing Great Lakes jobs and prints missing-file diagnostics instead of silently failing.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'scripts').exists()), PROJECT_CANDIDATES[0])
OUT_DIR = PROJECT_DIR / 'results' / 'poster_ablation_appendix'
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 240,
    'font.size': 15,
    'axes.titlesize': 17,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12,
    'figure.titlesize': 21,
    'axes.grid': True,
    'grid.alpha': 0.22,
    'axes.linewidth': 1.1,
})

print('PROJECT_DIR =', PROJECT_DIR)
print('OUT_DIR     =', OUT_DIR)


## 1. EMA Length Ablation

The EMA ablation asks whether sampling from raw checkpoint weights or from post-hoc EMA-smoothed weights changes sample quality.

Metrics used here:

- `hist_l1`: one-point PDF L1 distance to real data; lower is better.
- `pk_log10_mae`: mean absolute log10 error of generated mean \(P(k)\) against real mean \(P(k)\); lower is better.
- `std_ratio`: generated / real standard deviation; closer to 1 is better.
- `pk_ratio_low_k`, `pk_ratio_mid_k`, `pk_ratio_high_k`: broad-band power ratios; closer to 1 is better.

For the poster appendix, the key result is usually whether any EMA target clearly improves both one-point statistics and \(P(k)\), or whether the effect is small/tradeoff-like.


In [ ]:
def find_ema_metric_files(project_dir: Path) -> list[Path]:
    patterns = [
        'results/nf_generalize_fig2/quickcheck/*ema_metrics.csv',
        'results/nf_sweep_ema_sigma/quickcheck/*ema*metrics*.csv',
        'results/nf_sweep_small_ema/quickcheck/*ema*metrics*.csv',
        'results/nf_sweep_v2/quickcheck/*ema*metrics*.csv',
    ]
    files: list[Path] = []
    for pat in patterns:
        files.extend(project_dir.glob(pat))
    return sorted(set(files))

ema_files = find_ema_metric_files(PROJECT_DIR)
print(f'found {len(ema_files)} EMA metric files')
for p in ema_files:
    print(' -', p.relative_to(PROJECT_DIR))

if not ema_files:
    display(Markdown('''**No EMA metrics found yet.** On Great Lakes, first run the dedicated EMA notebook/script, e.g. `notebooks/nf_generalize_fig2_u64_d2p15_ema_check.ipynb`, then rerun this notebook.'''))


In [ ]:
def read_ema_metrics(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['source_file'] = str(path.relative_to(PROJECT_DIR)) if path.is_relative_to(PROJECT_DIR) else str(path)
    if 'ema_value' not in df.columns and 'ema_label' in df.columns:
        def parse_ema(label):
            if str(label) == 'raw':
                return np.nan
            s = str(label).replace('ema', '').replace('p', '.')
            try:
                return float(s)
            except Exception:
                return np.nan
        df['ema_value'] = df['ema_label'].map(parse_ema)
    return df

ema_tables = []
for path in ema_files:
    try:
        df = read_ema_metrics(path)
        needed = {'ema_label', 'hist_l1', 'pk_log10_mae'}
        if needed.issubset(df.columns):
            ema_tables.append(df)
        else:
            print('Skipping because required columns are missing:', path, sorted(df.columns))
    except Exception as exc:
        print('Failed reading', path, repr(exc))

if ema_tables:
    ema_all = pd.concat(ema_tables, ignore_index=True)
    display(ema_all.head())
else:
    ema_all = pd.DataFrame()


In [ ]:
def plot_one_ema_table(df: pd.DataFrame, title: str, out_name: str) -> None:
    plot_df = df.copy()
    plot_df['x'] = plot_df['ema_value']
    if 'ema_label' in plot_df:
        raw = plot_df[plot_df['ema_label'].astype(str) == 'raw']
        nonraw = plot_df[plot_df['ema_label'].astype(str) != 'raw'].sort_values('ema_value')
    else:
        raw = plot_df.iloc[0:0]
        nonraw = plot_df.sort_values('ema_value')

    fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8), constrained_layout=True)
    panels = [
        ('hist_l1', 'One-point PDF L1', 'lower is better'),
        ('pk_log10_mae', r'$P(k)$ log10 MAE', 'lower is better'),
        ('std_ratio', 'Generated / real std', 'target = 1'),
    ]
    for ax, (col, ylabel, subtitle) in zip(axes, panels):
        if col not in plot_df.columns:
            ax.set_visible(False)
            continue
        if len(nonraw):
            ax.plot(nonraw['ema_value'], nonraw[col], marker='o', lw=2.5, color='#1f77b4', label='EMA target')
        if len(raw):
            x_raw = float(np.nanmin(nonraw['ema_value'])) * 0.5 if len(nonraw) else 0.0
            ax.scatter([x_raw], [float(raw.iloc[0][col])], marker='*', s=170, color='black', label='raw')
            ax.annotate('raw', (x_raw, float(raw.iloc[0][col])), xytext=(5, 6), textcoords='offset points')
        if col == 'std_ratio':
            ax.axhline(1.0, color='black', ls='--', lw=1.5, alpha=0.75)
        else:
            best = plot_df.dropna(subset=[col]).sort_values(col).head(1)
            if len(best):
                bx = float(best.iloc[0]['ema_value']) if pd.notna(best.iloc[0]['ema_value']) else (float(np.nanmin(nonraw['ema_value'])) * 0.5 if len(nonraw) else 0.0)
                by = float(best.iloc[0][col])
                ax.scatter([bx], [by], marker='D', s=90, color='#d62728', zorder=5, label='best')
        ax.set_xlabel('EMA sigma_rel')
        ax.set_ylabel(ylabel)
        ax.set_title(subtitle)
        ax.legend(frameon=False)
    fig.suptitle(title, y=1.05)
    out = OUT_DIR / out_name
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)

if len(ema_all):
    for source_file, sub in ema_all.groupby('source_file', sort=False):
        clean = Path(source_file).stem.replace('_metrics', '')
        plot_one_ema_table(sub, f'EMA ablation: {clean}', f'{clean}_ema_summary.png')
else:
    print('No EMA table available to plot.')


In [ ]:
if len(ema_all):
    summary_rows = []
    for source_file, sub in ema_all.groupby('source_file'):
        row = {'source_file': source_file}
        for metric in ['hist_l1', 'pk_log10_mae']:
            if metric in sub.columns:
                best = sub.sort_values(metric).iloc[0]
                row[f'best_{metric}_label'] = best.get('ema_label')
                row[f'best_{metric}'] = float(best[metric])
        summary_rows.append(row)
    ema_summary = pd.DataFrame(summary_rows)
    display(ema_summary)
    out = OUT_DIR / 'ema_best_metric_summary.csv'
    ema_summary.to_csv(out, index=False)
    print('wrote', out)


## 2. CFG Dropout and Guidance-Scale Ablation

This section reads the continuous-cosmology CFG/guidance sweep scored with the **best VGG cosmology encoder**:

```text
normalized HI slice -> copy to 3 channels -> resize to 224 x 224
-> frozen VGG16 features -> avg+max pooling -> MLP(1024,512,256)
-> recovered cosmology
```

Two knobs are involved:

- `cfg_dropout`: training-time probability of dropping the conditioning vector, so the model learns both conditional and unconditional denoising.
- `guidance_scale`: sampling-time classifier-free guidance strength. `0` below means no guidance / `None`; larger values push samples harder toward the requested conditioning vector.

The expected table is now the VGG-scored sweep:

```text
results/nf_conditional_bias_probe_cfg_sweep/calibration_vgg/bias_probe_regime_slopes.csv
```

The main scalar here is the fitted calibration slope:

\[
\hat\theta_{\rm recovered} = a + b\,\theta_{\rm requested}.
\]

A slope closer to 1 means stronger tracking of the requested cosmology. A slope near 0 means the generated fields regress toward a mean cosmology. Guidance can increase the slope because it amplifies the conditional part of the denoising prediction, but too much guidance can also hurt realism or diversity, so it is an ablation rather than an automatic improvement.


In [ ]:
def find_guidance_slope_files(project_dir: Path) -> list[Path]:
    # Prefer the VGG-scored CFG sweep.  That is the version to use in the
    # poster because it uses the best trained VGG16 avg+max + MLP(1024,512,256) probe.
    preferred = sorted(set(project_dir.glob(
        'results/nf_conditional_bias_probe_cfg_sweep/calibration_vgg/bias_probe_regime_slopes.csv'
    )))
    if preferred:
        return preferred

    fallback = sorted(set(project_dir.glob(
        'results/nf_conditional_bias_probe_cfg_sweep/calibration*/bias_probe_regime_slopes.csv'
    )))
    if fallback:
        print('WARNING: VGG-scored CFG sweep table is missing; showing fallback tables instead.')
    return fallback

guidance_files = find_guidance_slope_files(PROJECT_DIR)
print(f'found {len(guidance_files)} CFG/guidance slope files')
for p in guidance_files:
    print(' -', p.relative_to(PROJECT_DIR))

if not guidance_files:
    display(Markdown('''**No VGG-scored CFG/guidance calibration slopes found yet.** Expected `results/nf_conditional_bias_probe_cfg_sweep/calibration_vgg/bias_probe_regime_slopes.csv`.'''))


In [ ]:
def infer_encoder_label_from_source(path: Path | str) -> str:
    s = str(path)
    if 'calibration_vgg' in s or 'vgg' in s.lower():
        return 'best VGG16 avg+max + MLP(1024,512,256)'
    if 'calibration_mlp' in s or 'mlp' in s.lower():
        return 'PCA+MLP probe'
    if 'nf_conditional_bias_probe_cfg_sweep' in s:
        return 'PCA+Ridge probe (fallback, not poster)'
    return 'cosmology probe'

guidance_tables = []
for path in guidance_files:
    try:
        gdf = pd.read_csv(path)
        rel = str(path.relative_to(PROJECT_DIR)) if path.is_relative_to(PROJECT_DIR) else str(path)
        gdf['source_file'] = rel
        gdf['encoder_probe'] = infer_encoder_label_from_source(rel)
        guidance_tables.append(gdf)
    except Exception as exc:
        print('failed reading', path, repr(exc))

if guidance_tables:
    guidance_all = pd.concat(guidance_tables, ignore_index=True)
    display(guidance_all.head())
else:
    guidance_all = pd.DataFrame()


In [ ]:
def guidance_to_float(label) -> float:
    s = str(label).lower()
    if s in {'noguidance', 'none', 'nan'}:
        return 0.0
    if s.startswith('g'):
        s = s[1:].replace('p', '.')
    try:
        return float(s)
    except Exception:
        return np.nan

def guidance_display(value: float) -> str:
    if not np.isfinite(value) or abs(value) < 1e-12:
        return 'guidance=None'
    return f'guidance={value:g}'

if len(guidance_all):
    df = guidance_all.copy()
    if 'guidance_label' not in df.columns:
        df['guidance_label'] = 'unknown'
    df['guidance_value'] = df['guidance_label'].map(guidance_to_float)
    df['guidance_display'] = df['guidance_value'].map(guidance_display)
    if 'cfg_dropout' not in df.columns:
        df['cfg_dropout'] = np.nan
    if 'encoder_probe' not in df.columns:
        df['encoder_probe'] = df['source_file'].map(infer_encoder_label_from_source)
    focus_params = ['Omega_m', 'sigma_8', 'A_SN1', 'A_SN2', 'A_AGN1', 'A_AGN2']
    df = df[df['parameter'].isin(focus_params)] if 'parameter' in df.columns else df

    keep_cols = [c for c in [
        'encoder_probe', 'regime', 'dataset_size', 'cfg_dropout', 'guidance_display',
        'parameter', 'slope', 'slope_ci16', 'slope_ci84', 'intercept', 'n_heldout', 'source_file'
    ] if c in df.columns]
    top = df.sort_values(['encoder_probe', 'regime', 'cfg_dropout', 'guidance_value', 'parameter'])
    display(top[keep_cols])
    out = OUT_DIR / 'guidance_slope_table.csv'
    top.to_csv(out, index=False)
    print('wrote', out)
else:
    df = pd.DataFrame()


In [ ]:
if len(guidance_all) and {'parameter', 'slope', 'guidance_value'}.issubset(df.columns):
    params = ['Omega_m', 'sigma_8', 'A_SN1', 'A_SN2']
    present = [p for p in params if p in set(df['parameter'])]
    fig, axes = plt.subplots(1, len(present), figsize=(5.8 * max(1, len(present)), 5.2), sharey=False, constrained_layout=True)
    if len(present) == 1:
        axes = [axes]
    colors = {'memorization': '#d62728', 'generalization': '#1f77b4'}
    markers = {'memorization': 'o', 'generalization': 's'}
    for ax, param in zip(axes, present):
        sub = df[df['parameter'] == param].copy()
        group_cols = ['encoder_probe', 'regime', 'cfg_dropout']
        for (encoder, regime, cfg), g in sub.groupby(group_cols, dropna=False):
            g = g.sort_values('guidance_value')
            cfg_text = 'cfg_dropout=?' if pd.isna(cfg) else f'cfg_dropout={float(cfg):g}'
            label = f'{regime}, {cfg_text}, {encoder}'
            ax.plot(
                g['guidance_value'], g['slope'],
                marker=markers.get(str(regime), 'o'), lw=3.0, ms=8.0,
                color=colors.get(str(regime), None), label=label,
            )
            if {'slope_ci16', 'slope_ci84'}.issubset(g.columns):
                ax.fill_between(g['guidance_value'], g['slope_ci16'], g['slope_ci84'], color=colors.get(str(regime), None), alpha=0.14)
            for _, row in g.iterrows():
                ax.annotate(
                    guidance_display(float(row['guidance_value'])).replace('guidance=', 'g='),
                    (row['guidance_value'], row['slope']), textcoords='offset points', xytext=(4, 5),
                    fontsize=9, color=colors.get(str(regime), '0.25'),
                )
        ax.axhline(1.0, color='black', ls='--', lw=1.5, label='ideal slope' if param == present[0] else None)
        ax.axhline(0.0, color='gray', ls=':', lw=1.2)
        ax.set_xlabel('Guidance scale at sampling')
        ax.set_ylabel('Recovered-vs-input slope')
        ax.set_title(param)
    handles, labels = axes[0].get_legend_handles_labels()
    # De-duplicate legend entries across panels.
    dedup = dict(zip(labels, handles))
    fig.legend(dedup.values(), dedup.keys(), loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=2, frameon=False, fontsize=11)
    fig.suptitle('CFG/guidance ablation: calibration slope', y=1.18)
    out = OUT_DIR / 'guidance_slope_summary.png'
    fig.savefig(out, bbox_inches='tight')
    plt.show()
    print('wrote', out)
else:
    print('No guidance slope table with the required columns was found.')


In [ ]:
# Show saved per-guidance calibration figures from the VGG-scored CFG sweep.
# Each PNG is one guidance setting.  Check the filename suffix:
#   noguidance = guidance_scale None; g1 = 1.0; g1p5 = 1.5; g2 = 2.0.
calib_pngs = sorted(PROJECT_DIR.glob('results/nf_conditional_bias_probe_cfg_sweep/calibration_vgg/bias_probe_calibration_recovered_vs_input*.png'))
if not calib_pngs:
    print('WARNING: no VGG-scored CFG PNGs found; falling back to any CFG calibration PNGs.')
    calib_pngs = sorted(PROJECT_DIR.glob('results/nf_conditional_bias_probe_cfg_sweep/calibration*/bias_probe_calibration_recovered_vs_input*.png'))
print(f'found {len(calib_pngs)} guidance calibration PNGs')
for p in calib_pngs[:12]:
    print(' -', p.relative_to(PROJECT_DIR))
    display(Image(filename=str(p)))
if len(calib_pngs) > 12:
    print('... truncated display; see directory for all PNGs')


## Great Lakes Commands If Outputs Are Missing

Use these only if the notebook reports missing files.

### EMA focused check

For the focused Fig. 2 EMA check:

```bash
cd /home/jiamingp/diffusion_models_repo
RUN_NAME=nf_fig2_u64_d2p15_noaug_200k NUM_SAMPLES=128 OVERWRITE=0   sbatch -A huterer2 scripts/slurm/sample_nf_generalize_fig2_small_ema_one.sbatch
```

Then rerun `notebooks/nf_generalize_fig2_u64_d2p15_ema_check.ipynb` once to score the samples and write the metrics CSV.

### CFG/guidance sweep check

The CFG/guidance jobs you ran should write under:

```text
results/nf_conditional_bias_probe_cfg_sweep/calibration/
```

Expected final table:

```text
results/nf_conditional_bias_probe_cfg_sweep/calibration/bias_probe_regime_slopes.csv
```

If missing, check jobs/logs:

```bash
sacct --starttime=2026-06-01 -u jiamingp --format=JobID,JobName%30,State,ExitCode,Elapsed | grep nf_bias_cfg
ls -lh results/nf_conditional_bias_probe_cfg_sweep/calibration
```


## Poster Interpretation Template

Use this wording if the ablation results are not the main story:

> We also checked a small set of sampling/training ablations. EMA smoothing and guidance scale changed the diagnostics quantitatively, but the main qualitative conclusions were stable: low-data models show memorization/collapse behavior, while the larger-data conditional model tracks the requested cosmology more clearly, especially for \(\Omega_m\). These ablations are shown as appendix checks rather than the main result.
